# Paso 6 - Casos de usuario explicados

Este notebook genera ejemplos interpretables del tipo:

> Para el usuario Z, el modelo recomienda X porque su historial tiene caracteristicas W.

La idea es comparar recomendaciones de distintos modelos para usuarios concretos y explicar el comportamiento del modelo adaptativo jerarquico.

In [1]:
from pathlib import Path
import ast
import pickle
import sys

import pandas as pd

PROJECT_DIR = Path('..').resolve().parent
SRC_DIR = PROJECT_DIR / 'H3' / 'src'
OUTPUT_DIR = PROJECT_DIR / 'H3' / 'outputs'
CACHE_DIR = PROJECT_DIR / 'H3' / 'cache'
sys.path.insert(0, str(SRC_DIR))

from step1_metrics_examples import category_diversity_at_k, novelty_at_k

K = 10
pd.set_option('display.max_colwidth', 220)

## 1. Cargar artefactos de analisis

Usamos el cache generado para el Paso 4. Este cache evita reconstruir componentes secuenciales, categoricos, jerarquicos y perfiles de usuario.

In [2]:
with (CACHE_DIR / 'step4_analysis_artifacts_sample_50000.pkl').open('rb') as file:
    artifacts = pickle.load(file)

with (CACHE_DIR / 'item_to_category.pkl').open('rb') as file:
    item_to_category = pickle.load(file)
with (CACHE_DIR / 'global_popularity.pkl').open('rb') as file:
    fallback_items, item_prob, _ = pickle.load(file)
with (CACHE_DIR / 'catalog_items.pkl').open('rb') as file:
    catalog_items = pickle.load(file)

catalog_size = len(catalog_items)

components_by_user = artifacts['components_by_user']
tuned_by_user = artifacts['tuned_by_user']
profiles_by_user = artifacts['profiles_by_user']

len(artifacts['eval_users'])

50000

## 2. Funciones para explicar recomendaciones

Para cada usuario mostramos:

- historial reciente,
- categoria dominante/ultima categoria conocida,
- confianza secuencial,
- pesos adaptativos,
- recomendacion de cada componente,
- recomendacion final,
- item real de test,
- explicacion textual del comportamiento.

In [3]:
def fixed_hybrid_from_components(user):
    components = components_by_user[user]
    scores = {}
    for rank, item in enumerate(components['seq'][:200], start=1):
        scores[int(item)] = scores.get(int(item), 0.0) + 0.70 / rank
    for rank, item in enumerate(components['cat'][:200], start=1):
        scores[int(item)] = scores.get(int(item), 0.0) + 0.30 / rank
    for rank, item in enumerate(fallback_items[:200], start=1):
        scores[int(item)] = scores.get(int(item), 0.0) + 0.05 / rank
    ranked = [item for item, _ in sorted(scores.items(), key=lambda pair: (-pair[1], pair[0]))]
    return ranked[:K]


def recommendation_categories(items):
    return [item_to_category.get(int(item)) for item in items]


def summarize_history(history):
    if not history:
        return 'Sin historial disponible.'
    parts = []
    for event in history:
        parts.append(f"{event['event']} item {event['itemid']} cat {event['categoryid']}")
    return ' | '.join(parts)


def profile_type(profile):
    h = profile['history_length']
    conf = profile['transition_confidence']
    if h <= 2 and conf < 0.4:
        return 'historial corto y baja confianza secuencial'
    if h <= 2:
        return 'historial corto'
    if conf >= 0.7:
        return 'historial mayor y alta confianza secuencial'
    return 'historial mayor con confianza secuencial moderada/baja'


def explain_case(user):
    profile = profiles_by_user[user]
    components = components_by_user[user]
    seq = components['seq'][:K]
    cat = components['cat'][:K]
    hier = components['hier'][:K]
    fixed = fixed_hybrid_from_components(user)
    tuned = tuned_by_user[user]
    true_item = profile['true_item']
    weights = profile['weights']
    recent_categories = [h['categoryid'] for h in profile['recent_history'] if h['categoryid'] is not None]
    last_category = recent_categories[-1] if recent_categories else None
    top_tuned_categories = recommendation_categories(tuned)
    explanation = (
        f"Usuario con {profile_type(profile)}. "
        f"Su ultima categoria conocida es {last_category}. "
        f"La confianza secuencial es {profile['transition_confidence']:.3f}; por eso el modelo asigna "
        f"peso seq={weights['seq']:.2f}, categoria={weights['category']:.2f}, "
        f"jerarquia={weights['hierarchy']:.2f}, popularidad={weights['popularity']:.2f}. "
        f"El ranking final mezcla items sugeridos por transiciones recientes con items de categorias cercanas/ancestros. "
        f"El item real de test es {true_item} (categoria {profile['true_category']})."
    )
    return {
        'visitorid': user,
        'tipo_usuario': profile_type(profile),
        'history_length': profile['history_length'],
        'transition_confidence': profile['transition_confidence'],
        'peso_seq': weights['seq'],
        'peso_categoria': weights['category'],
        'peso_jerarquia': weights['hierarchy'],
        'peso_popularidad': weights['popularity'],
        'historial_reciente': summarize_history(profile['recent_history']),
        'ultima_categoria': last_category,
        'true_item': true_item,
        'true_category': profile['true_category'],
        'sequential_top10': seq,
        'sequential_categories': recommendation_categories(seq),
        'category_top10': cat,
        'category_categories': recommendation_categories(cat),
        'hierarchical_top10': hier,
        'hierarchical_categories': recommendation_categories(hier),
        'fixed_hybrid_top10': fixed,
        'fixed_hybrid_categories': recommendation_categories(fixed),
        'tuned_top10': tuned,
        'tuned_categories': top_tuned_categories,
        'sequential_hit': int(true_item in seq),
        'category_hit': int(true_item in cat),
        'hierarchical_hit': int(true_item in hier),
        'fixed_hybrid_hit': int(true_item in fixed),
        'tuned_hit': int(true_item in tuned),
        'tuned_novelty@10': novelty_at_k(tuned, item_prob, catalog_size, K),
        'tuned_diversity@10': category_diversity_at_k(tuned, item_to_category, K),
        'explicacion': explanation,
    }

## 3. Seleccionar casos representativos

Seleccionamos usuarios que ilustren distintos comportamientos:

- historial corto con baja confianza secuencial,
- historial corto con acierto del modelo final,
- historial mayor con alta confianza secuencial,
- casos donde el modelo final acierta y otros componentes no.

In [4]:
cases = []
seen_types = set()

for user in artifacts['eval_users']:
    case = explain_case(user)
    key = None
    if case['history_length'] <= 2 and case['transition_confidence'] < 0.4 and case['tuned_hit']:
        key = 'corto_baja_conf_hit'
    elif case['history_length'] <= 2 and case['tuned_hit'] and not case['sequential_hit']:
        key = 'corto_tuned_gana_a_seq'
    elif case['history_length'] >= 5 and case['transition_confidence'] >= 0.7 and case['tuned_hit']:
        key = 'largo_alta_conf_hit'
    elif case['tuned_hit'] and not case['fixed_hybrid_hit']:
        key = 'tuned_gana_a_fixed'
    elif case['tuned_hit']:
        key = 'hit_general'

    if key is not None and key not in seen_types:
        case['caso'] = key
        cases.append(case)
        seen_types.add(key)
    if len(cases) >= 6:
        break

cases_df = pd.DataFrame(cases)
cases_df.to_csv(OUTPUT_DIR / 'step6_casos_usuario_explicados.csv', index=False)
display(cases_df[[
    'caso', 'visitorid', 'tipo_usuario', 'history_length', 'transition_confidence',
    'peso_seq', 'peso_categoria', 'peso_jerarquia', 'ultima_categoria',
    'true_item', 'true_category', 'sequential_hit', 'fixed_hybrid_hit', 'tuned_hit',
    'tuned_novelty@10', 'tuned_diversity@10', 'explicacion'
]])

,caso,visitorid,tipo_usuario,history_length,transition_confidence,peso_seq,peso_categoria,peso_jerarquia,ultima_categoria,true_item,true_category,sequential_hit,fixed_hybrid_hit,tuned_hit,tuned_novelty@10,tuned_diversity@10,explicacion
0,hit_general,513229,historial corto,1,0.812690,0.591646,0.325856,0.035336,1047,290993,484,1,1,1,14.170221,0.666667,"Usuario con historial corto. Su ultima categoria conocida es 1047. La confianza secuencial es 0.813; por eso el modelo asigna peso seq=0.59, categoria=0.33, jerarquia=0.04, popularidad=0.05. El ranking final mezcla i..."
1,corto_baja_conf_hit,1343397,historial corto y baja confianza secuencial,2,0.208015,0.333838,0.461172,0.155808,389,239766,996,1,1,1,15.215832,0.666667,"Usuario con historial corto y baja confianza secuencial. Su ultima categoria conocida es 389. La confianza secuencial es 0.208; por eso el modelo asigna peso seq=0.33, categoria=0.46, jerarquia=0.16, popularidad=0.05..."
2,largo_alta_conf_hit,904138,historial mayor y alta confianza secuencial,8,1.000000,0.731720,0.224483,0.000000,29,231243,29,1,1,1,14.679909,0.377778,"Usuario con historial mayor y alta confianza secuencial. Su ultima categoria conocida es 29. La confianza secuencial es 1.000; por eso el modelo asigna peso seq=0.73, categoria=0.22, jerarquia=0.00, popularidad=0.04...."
3,corto_tuned_gana_a_seq,489800,historial corto,2,0.965990,0.674947,0.273056,0.006227,535,198400,535,0,1,1,14.746685,0.200000,"Usuario con historial corto. Su ultima categoria conocida es 535. La confianza secuencial es 0.966; por eso el modelo asigna peso seq=0.67, categoria=0.27, jerarquia=0.01, popularidad=0.05. El ranking final mezcla it..."
4,tuned_gana_a_fixed,697169,historial mayor con confianza secuencial moderada/baja,8,0.416029,0.488140,0.357882,0.107820,1085,86824,1085,0,0,1,13.170080,0.777778,"Usuario con historial mayor con confianza secuencial moderada/baja. Su ultima categoria conocida es 1085. La confianza secuencial es 0.416; por eso el modelo asigna peso seq=0.49, categoria=0.36, jerarquia=0.11, popu..."


## 4. Comparacion de rankings por usuario

La siguiente tabla muestra, para cada usuario, que recomienda cada modelo y en que categorias caen esas recomendaciones.

In [5]:
ranking_rows = []
for case in cases:
    for model_name, items_key, cats_key in [
        ('Sequential Transition', 'sequential_top10', 'sequential_categories'),
        ('Category Popularity', 'category_top10', 'category_categories'),
        ('Hierarchical Category', 'hierarchical_top10', 'hierarchical_categories'),
        ('Fixed Hybrid', 'fixed_hybrid_top10', 'fixed_hybrid_categories'),
        ('Adaptive Hierarchical Hybrid', 'tuned_top10', 'tuned_categories'),
    ]:
        ranking_rows.append({
            'caso': case['caso'],
            'visitorid': case['visitorid'],
            'model': model_name,
            'recommendations': case[items_key],
            'recommendation_categories': case[cats_key],
            'true_item': case['true_item'],
            'true_category': case['true_category'],
            'hit@10': int(case['true_item'] in case[items_key]),
        })

rankings_df = pd.DataFrame(ranking_rows)
rankings_df.to_csv(OUTPUT_DIR / 'step6_comparacion_rankings_por_usuario.csv', index=False)
display(rankings_df)

,caso,visitorid,model,recommendations,recommendation_categories,true_item,true_category,hit@10
0,hit_general,513229,Sequential Transition,"[126236, 340490, 290993, 406795, 463165, 20249, 108032, 233974, 366009, 56498]","[1047, 50, 484, 1047, 1047, 884, 1355, 1355, 1155, 50]",290993,484,1
1,hit_general,513229,Category Popularity,"[126236, 6692, 297513, 109924, 2992, 404591, 136524, 321946, 76477, 393809]","[1047, 1047, 1047, 1047, 1047, 1047, 1047, 1047, 1047, 1047]",290993,484,0
2,hit_general,513229,Hierarchical Category,"[126236, 6692, 297513, 109924, 404591, 2992, 136524, 321946, 76477, 393809]","[1047, 1047, 1047, 1047, 1047, 1047, 1047, 1047, 1047, 1047]",290993,484,0
3,hit_general,513229,Fixed Hybrid,"[126236, 340490, 290993, 406795, 463165, 6692, 20249, 461686, 108032, 297513]","[1047, 50, 484, 1047, 1047, 1047, 884, 1037, 1355, 1047]",290993,484,1
4,hit_general,513229,Adaptive Hierarchical Hybrid,"[126236, 340490, 290993, 6692, 406795, 463165, 297513, 20249, 461686, 109924]","[1047, 50, 484, 1047, 1047, 1047, 1047, 884, 1037, 1047]",290993,484,1
5,corto_baja_conf_hit,1343397,Sequential Transition,"[239766, 461686, 257040, 119736, 320130, 9877, 309778, 312728, 384302, 7943]","[996, 1037, 683, 57, 1483, 858, 683, 1098, 5, 398]",239766,996,1
6,corto_baja_conf_hit,1343397,Category Popularity,"[92214, 120644, 283699, 51656, 200978, 52013, 83723, 372273, 19996, 124503]","[389, 389, 389, 389, 389, 389, 389, 389, 389, 389]",239766,996,0
7,corto_baja_conf_hit,1343397,Hierarchical Category,"[283699, 120644, 92214, 200978, 51656, 52013, 83723, 372273, 124503, 424559]","[389, 389, 389, 389, 389, 389, 389, 389, 389, 389]",239766,996,0
8,corto_baja_conf_hit,1343397,Fixed Hybrid,"[239766, 461686, 92214, 257040, 119736, 320130, 120644, 9877, 309778, 283699]","[996, 1037, 389, 683, 57, 1483, 389, 858, 683, 389]",239766,996,1
9,corto_baja_conf_hit,1343397,Adaptive Hierarchical Hybrid,"[92214, 239766, 283699, 120644, 461686, 51656, 257040, 200978, 119736, 52013]","[389, 996, 389, 389, 1037, 389, 683, 389, 57, 389]",239766,996,1


## 5. Explicaciones comparativas por modelo

Esta seccion genera textos del tipo: "Para el usuario Z, el modelo Y recomienda X porque su historial tiene caracteristicas W".

La idea es que la comparacion cualitativa no dependa solo del modelo final, sino que explique que prioriza cada baseline y por que cambian los rankings.

In [6]:
def model_reason(case, model_name, recommendations, categories):
    top_item = recommendations[0] if recommendations else None
    top_category = categories[0] if categories else None
    true_item = case['true_item']
    hit = int(true_item in recommendations)
    last_cat = case['ultima_categoria']
    hlen = case['history_length']
    conf = case['transition_confidence']

    if model_name == 'Sequential Transition':
        reason = (
            f"prioriza items que historicamente aparecen despues del ultimo item observado por el usuario. "
            f"Como la confianza secuencial del usuario es {conf:.3f}, esta senal es {'fuerte' if conf >= 0.7 else 'moderada/baja'}."
        )
    elif model_name == 'Category Popularity':
        reason = (
            f"prioriza items populares dentro de la ultima categoria conocida del usuario ({last_cat}). "
            f"No usa directamente transiciones item-item ni ancestros de categoria."
        )
    elif model_name == 'Hierarchical Category':
        reason = (
            f"parte desde la ultima categoria conocida ({last_cat}) y usa backoff hacia categorias ancestras si necesita mas candidatos. "
            f"Por eso tiende a recomendar items de la misma rama categorica."
        )
    elif model_name == 'Fixed Hybrid':
        reason = (
            f"mezcla secuencia y categoria con pesos fijos para todos los usuarios. "
            f"No cambia su comportamiento aunque el usuario tenga historial corto ({hlen} eventos) o distinta confianza secuencial."
        )
    elif model_name == 'Adaptive Hierarchical Hybrid':
        reason = (
            f"ajusta los pesos al perfil del usuario: seq={case['peso_seq']:.2f}, categoria={case['peso_categoria']:.2f}, "
            f"jerarquia={case['peso_jerarquia']:.2f}, popularidad={case['peso_popularidad']:.2f}. "
            f"Con historial de {hlen} eventos y confianza secuencial {conf:.3f}, combina transiciones recientes con categoria/jerarquia."
        )
    else:
        reason = 'modelo no reconocido.'

    return (
        f"Para el usuario {case['visitorid']}, {model_name} recomienda primero el item {top_item} "
        f"(categoria {top_category}) porque {reason} "
        f"El item real de test es {true_item} (categoria {case['true_category']}); "
        f"hit@10={hit}."
    )


explanation_rows = []
for case in cases:
    model_specs = [
        ('Sequential Transition', case['sequential_top10'], case['sequential_categories']),
        ('Category Popularity', case['category_top10'], case['category_categories']),
        ('Hierarchical Category', case['hierarchical_top10'], case['hierarchical_categories']),
        ('Fixed Hybrid', case['fixed_hybrid_top10'], case['fixed_hybrid_categories']),
        ('Adaptive Hierarchical Hybrid', case['tuned_top10'], case['tuned_categories']),
    ]
    for model_name, recs, cats in model_specs:
        explanation_rows.append({
            'caso': case['caso'],
            'visitorid': case['visitorid'],
            'model': model_name,
            'history_length': case['history_length'],
            'transition_confidence': case['transition_confidence'],
            'last_category': case['ultima_categoria'],
            'top_item': recs[0] if recs else None,
            'top_category': cats[0] if cats else None,
            'true_item': case['true_item'],
            'true_category': case['true_category'],
            'hit@10': int(case['true_item'] in recs),
            'explanation': model_reason(case, model_name, recs, cats),
        })

model_explanations = pd.DataFrame(explanation_rows)
model_explanations.to_csv(OUTPUT_DIR / 'step6_explicaciones_por_modelo.csv', index=False)
display(model_explanations)

,caso,visitorid,model,history_length,transition_confidence,last_category,top_item,top_category,true_item,true_category,hit@10,explanation
0,hit_general,513229,Sequential Transition,1,0.812690,1047,126236,1047,290993,484,1,"Para el usuario 513229, Sequential Transition recomienda primero el item 126236 (categoria 1047) porque prioriza items que historicamente aparecen despues del ultimo item observado por el usuario. Como la confianza s..."
1,hit_general,513229,Category Popularity,1,0.812690,1047,126236,1047,290993,484,0,"Para el usuario 513229, Category Popularity recomienda primero el item 126236 (categoria 1047) porque prioriza items populares dentro de la ultima categoria conocida del usuario (1047). No usa directamente transicion..."
2,hit_general,513229,Hierarchical Category,1,0.812690,1047,126236,1047,290993,484,0,"Para el usuario 513229, Hierarchical Category recomienda primero el item 126236 (categoria 1047) porque parte desde la ultima categoria conocida (1047) y usa backoff hacia categorias ancestras si necesita mas candida..."
3,hit_general,513229,Fixed Hybrid,1,0.812690,1047,126236,1047,290993,484,1,"Para el usuario 513229, Fixed Hybrid recomienda primero el item 126236 (categoria 1047) porque mezcla secuencia y categoria con pesos fijos para todos los usuarios. No cambia su comportamiento aunque el usuario tenga..."
4,hit_general,513229,Adaptive Hierarchical Hybrid,1,0.812690,1047,126236,1047,290993,484,1,"Para el usuario 513229, Adaptive Hierarchical Hybrid recomienda primero el item 126236 (categoria 1047) porque ajusta los pesos al perfil del usuario: seq=0.59, categoria=0.33, jerarquia=0.04, popularidad=0.05. Con h..."
5,corto_baja_conf_hit,1343397,Sequential Transition,2,0.208015,389,239766,996,239766,996,1,"Para el usuario 1343397, Sequential Transition recomienda primero el item 239766 (categoria 996) porque prioriza items que historicamente aparecen despues del ultimo item observado por el usuario. Como la confianza s..."
6,corto_baja_conf_hit,1343397,Category Popularity,2,0.208015,389,92214,389,239766,996,0,"Para el usuario 1343397, Category Popularity recomienda primero el item 92214 (categoria 389) porque prioriza items populares dentro de la ultima categoria conocida del usuario (389). No usa directamente transiciones..."
7,corto_baja_conf_hit,1343397,Hierarchical Category,2,0.208015,389,283699,389,239766,996,0,"Para el usuario 1343397, Hierarchical Category recomienda primero el item 283699 (categoria 389) porque parte desde la ultima categoria conocida (389) y usa backoff hacia categorias ancestras si necesita mas candidat..."
8,corto_baja_conf_hit,1343397,Fixed Hybrid,2,0.208015,389,239766,996,239766,996,1,"Para el usuario 1343397, Fixed Hybrid recomienda primero el item 239766 (categoria 996) porque mezcla secuencia y categoria con pesos fijos para todos los usuarios. No cambia su comportamiento aunque el usuario tenga..."
9,corto_baja_conf_hit,1343397,Adaptive Hierarchical Hybrid,2,0.208015,389,92214,389,239766,996,1,"Para el usuario 1343397, Adaptive Hierarchical Hybrid recomienda primero el item 92214 (categoria 389) porque ajusta los pesos al perfil del usuario: seq=0.33, categoria=0.46, jerarquia=0.16, popularidad=0.05. Con hi..."


## 6. Explicaciones textuales caso a caso

In [7]:
for case in cases:
    print('=' * 100)
    print(f"Caso: {case['caso']} | Usuario: {case['visitorid']}")
    print(case['explicacion'])
    print(f"Historial: {case['historial_reciente']}")
    print(f"Sequential top-10: {case['sequential_top10']}")
    print(f"Fixed Hybrid top-10: {case['fixed_hybrid_top10']}")
    print(f"Adaptive Hierarchical Hybrid top-10: {case['tuned_top10']}")
    print(f"Item real: {case['true_item']} | categoria real: {case['true_category']}")
    print(f"Hits -> seq: {case['sequential_hit']}, fixed: {case['fixed_hybrid_hit']}, tuned: {case['tuned_hit']}")

Caso: hit_general | Usuario: 513229
Usuario con historial corto. Su ultima categoria conocida es 1047. La confianza secuencial es 0.813; por eso el modelo asigna peso seq=0.59, categoria=0.33, jerarquia=0.04, popularidad=0.05. El ranking final mezcla items sugeridos por transiciones recientes con items de categorias cercanas/ancestros. El item real de test es 290993 (categoria 484).
Historial: view item 264876 cat 1047
Sequential top-10: [126236, 340490, 290993, 406795, 463165, 20249, 108032, 233974, 366009, 56498]
Fixed Hybrid top-10: [126236, 340490, 290993, 406795, 463165, 6692, 20249, 461686, 108032, 297513]
Adaptive Hierarchical Hybrid top-10: [126236, 340490, 290993, 6692, 406795, 463165, 297513, 20249, 461686, 109924]
Item real: 290993 | categoria real: 484
Hits -> seq: 1, fixed: 1, tuned: 1
Caso: corto_baja_conf_hit | Usuario: 1343397
Usuario con historial corto y baja confianza secuencial. Su ultima categoria conocida es 389. La confianza secuencial es 0.208; por eso el modelo